# RetainIQ — Phase 7.2: Revenue-at-Risk Prioritization

## Objective

I convert exposure into a prioritization framework for business action.

For segments, I use the required formula:

**Priority Score = Revenue at Risk × Ease-of-Intervention Score**

For geographies, I use the same formula only when a geography-specific score has been supplied. Otherwise, I keep the geography view based on revenue at risk alone instead of introducing an unsupported assumption.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

ROOT = Path("..")
OUTPUT_DIR = ROOT / "outputs"
CONFIG_DIR = ROOT / "config"


def print_result(message):
    print(f"Result: {message}")


In [5]:
segment_df = pd.read_csv(OUTPUT_DIR / "phase_07_segment_business_base.csv")
state_df = pd.read_csv(OUTPUT_DIR / "phase_07_state_business_base.csv")
city_df = pd.read_csv(OUTPUT_DIR / "phase_07_city_business_base.csv")
segment_scores = pd.read_csv(CONFIG_DIR / "ease_of_intervention_scores.csv")
geo_score_path = CONFIG_DIR / "geography_ease_of_intervention_scores.csv"

def numeric_cast(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

segment_df = numeric_cast(segment_df, ["revenue_at_risk", "avg_cltv", "churn_rate_pct"])
state_df = numeric_cast(state_df, ["revenue_at_risk", "avg_cltv", "churn_rate_pct"])
city_df = numeric_cast(city_df, ["revenue_at_risk", "avg_cltv", "churn_rate_pct"])
segment_scores = numeric_cast(segment_scores, ["ease_of_intervention_score"])

print(f"Loaded {len(segment_df):,} segments, {len(state_df):,} states, and {len(city_df):,} city markets.")

Loaded 3 segments, 1 states, and 1,106 city markets.


### Result & conclusion

I loaded the Phase 7 business base tables and the editable intervention-score file. The next step is to validate the scores before I use them in any priority formula.

In [7]:
if "print_result" not in globals():
    def print_result(message):
        print(f"Result: {message}")

valid_score_mask = segment_scores["ease_of_intervention_score"].between(1, 5, inclusive="both")
missing_segments = segment_scores.loc[segment_scores["ease_of_intervention_score"].isna(), "segment_name"].tolist()
invalid_segments = segment_scores.loc[
    segment_scores["ease_of_intervention_score"].notna() & ~valid_score_mask,
    "segment_name"
].tolist()

if missing_segments or invalid_segments:
    print_result(
        f"Segment scoring is not ready. Missing scores: {len(missing_segments)}; invalid scores: {len(invalid_segments)}."
    )
    if missing_segments:
        print("Missing segment scores:", ", ".join(missing_segments))
    if invalid_segments:
        print("Invalid segment scores:", ", ".join(invalid_segments))
    raise ValueError(
        "Enter an integer ease_of_intervention_score from 1 to 5 for every segment in "
        "config/ease_of_intervention_scores.csv, then rerun this notebook."
    )

segment_scores["ease_of_intervention_score"] = segment_scores["ease_of_intervention_score"].astype(int)
print_result("All segment intervention scores are present and valid integers from 1 to 5.")
display(segment_scores)

Result: All segment intervention scores are present and valid integers from 1 to 5.


,segment_id,segment_name,retention_profile,ease_of_intervention_score,score_note
0,0,Segment 0 — Emerging Risk,Retention Priority,4,Enter a subjective integer from 1 to 5; 5 mean...
1,1,Segment 1 — High-Value Stable,Maintain Value,5,Enter a subjective integer from 1 to 5; 5 mean...
2,2,Segment 2 — High-Value At-Risk,Protect High-Value,3,Enter a subjective integer from 1 to 5; 5 mean...


### Result & conclusion

The weighted segment analysis is now based only on explicitly supplied business assumptions. No score is inferred from churn rate, CLTV, or any other analytical field.

In [8]:
segment_priority = segment_df.merge(
    segment_scores[["segment_id", "segment_name", "retention_profile", "ease_of_intervention_score"]],
    on=["segment_id", "segment_name", "retention_profile"],
    how="left",
    validate="one_to_one"
)

segment_priority["priority_score"] = (
    segment_priority["revenue_at_risk"] * segment_priority["ease_of_intervention_score"]
)
segment_priority["priority_basis"] = "Revenue at Risk × Ease of Intervention"

segment_priority = segment_priority.sort_values(
    ["priority_score", "revenue_at_risk"], ascending=[False, False]
).reset_index(drop=True)
segment_priority["priority_order"] = np.arange(1, len(segment_priority) + 1)

print_result(
    f"Calculated weighted priorities for {len(segment_priority)} segments. The largest weighted exposure is {segment_priority.iloc[0]['segment_name']}."
)
display(segment_priority)

Result: Calculated weighted priorities for 3 segments. The largest weighted exposure is Segment 0 — Emerging Risk.


,segment_id,segment_name,retention_profile,customers,share_pct,avg_tenure_months,avg_monthly_charge,avg_cltv,avg_satisfaction,churned_customers,churn_rate_pct,revenue_at_risk,ease_of_intervention_score,priority_score,priority_basis,priority_order
0,0,Segment 0 — Emerging Risk,Retention Priority,3226,45.80,15.74,69.78,4013.45,2.81,1516,46.99,1944581.19,4,7778324.76,Revenue at Risk × Ease of Intervention,1
1,2,Segment 2 — High-Value At-Risk,Protect High-Value,2260,32.09,57.24,87.39,4972.51,3.45,243,10.75,1696896.61,3,5090689.83,Revenue at Risk × Ease of Intervention,2
2,1,Segment 1 — High-Value Stable,Maintain Value,1557,22.11,30.80,21.53,4371.24,3.86,110,7.06,42982.02,5,214910.10,Revenue at Risk × Ease of Intervention,3


### Result & conclusion

Each segment now has a transparent priority score. The ordering combines **financial exposure** with the **business ease of intervention** that I explicitly provide, rather than relying on revenue at risk alone.

In [9]:
# State and city geography views.
state_priority = state_df.copy()
city_priority = city_df.copy()

if geo_score_path.exists():
    geo_scores = pd.read_csv(geo_score_path)
    geo_scores["ease_of_intervention_score"] = pd.to_numeric(
        geo_scores["ease_of_intervention_score"], errors="coerce"
    )
else:
    geo_scores = pd.DataFrame(columns=["state", "city", "ease_of_intervention_score"])

valid_geo_scores = geo_scores[
    geo_scores["ease_of_intervention_score"].between(1, 5, inclusive="both")
].copy()

city_priority = city_priority.merge(
    valid_geo_scores[["state", "city", "ease_of_intervention_score"]],
    on=["state", "city"], how="left", validate="one_to_one"
)

state_scores = (
    valid_geo_scores.groupby("state", as_index=False)["ease_of_intervention_score"]
    .mean()
    .rename(columns={"ease_of_intervention_score": "ease_of_intervention_score"})
)
state_priority = state_priority.merge(state_scores, on="state", how="left", validate="one_to_one")

for df in [state_priority, city_priority]:
    df["priority_score"] = np.where(
        df["ease_of_intervention_score"].notna(),
        df["revenue_at_risk"] * df["ease_of_intervention_score"],
        df["revenue_at_risk"]
    )
    df["priority_basis"] = np.where(
        df["ease_of_intervention_score"].notna(),
        "Revenue at Risk × Geography Ease of Intervention",
        "Revenue at Risk only"
    )

state_priority = state_priority.sort_values(["priority_score", "revenue_at_risk"], ascending=[False, False]).reset_index(drop=True)
city_priority = city_priority.sort_values(["priority_score", "revenue_at_risk"], ascending=[False, False]).reset_index(drop=True)
state_priority["priority_order"] = np.arange(1, len(state_priority) + 1)
city_priority["priority_order"] = np.arange(1, len(city_priority) + 1)

weighted_city_count = int(city_priority["ease_of_intervention_score"].notna().sum())
print_result(
    f"Prepared {len(state_priority)} states and {len(city_priority):,} city markets. {weighted_city_count:,} city markets have geography-specific intervention scores; the rest retain revenue-at-risk-only prioritization."
)
display(city_priority.head(20))

Result: Prepared 1 states and 1,106 city markets. 0 city markets have geography-specific intervention scores; the rest retain revenue-at-risk-only prioritization.


,state,city,customers,churned_customers,avg_cltv,total_revenue,avg_monthly_charge,avg_satisfaction,churn_rate_pct,revenue_at_risk,ease_of_intervention_score,priority_score,priority_basis,priority_order
0,California,San Diego,285,185,4345.09,738416.01,72.02,2.54,64.91,385446.39,NaN,385446.39,Revenue at Risk only,1
1,California,Los Angeles,293,78,4369.85,852725.23,62.13,3.21,26.62,147090.46,NaN,147090.46,Revenue at Risk only,2
2,California,Sacramento,108,26,4540.66,353371.84,62.73,3.25,24.07,59449.20,NaN,59449.20,Revenue at Risk only,3
3,California,San Francisco,104,31,4335.46,306995.99,64.64,3.25,29.81,56894.27,NaN,56894.27,Revenue at Risk only,4
4,California,Santa Barbara,28,10,4198.11,78526.26,62.84,3.21,35.71,46033.71,NaN,46033.71,Revenue at Risk only,5
5,California,San Jose,112,29,4233.38,326478.36,65.59,3.13,25.89,44393.49,NaN,44393.49,Revenue at Risk only,6
6,California,Temecula,38,22,4367.87,119268.27,76.28,2.53,57.89,44102.41,NaN,44102.41,Revenue at Risk only,7
7,California,Escondido,51,16,4388.86,155899.80,67.88,3.02,31.37,38869.89,NaN,38869.89,Revenue at Risk only,8
8,California,Fallbrook,43,26,4531.28,93409.28,69.83,2.72,60.47,34691.78,NaN,34691.78,Revenue at Risk only,9
9,California,Torrance,25,8,4606.20,77995.90,61.58,2.88,32.00,33534.79,NaN,33534.79,Revenue at Risk only,10


### Result & conclusion

Geography is handled conservatively. When a geography-specific ease score exists, I use the weighted formula; otherwise, I do not pretend that a segment-level score applies to a whole city or state.

In [10]:
combined_segment = segment_priority.assign(scope="segment", entity_name=segment_priority["segment_name"])
combined_city = city_priority.assign(
    scope="city",
    entity_name=city_priority["state"].astype(str) + " — " + city_priority["city"].astype(str)
)
combined_state = state_priority.assign(scope="state", entity_name=state_priority["state"].astype(str))

combined = pd.concat([
    combined_segment[["scope", "entity_name", "customers", "avg_cltv", "churn_rate_pct", "revenue_at_risk", "ease_of_intervention_score", "priority_score", "priority_basis"]],
    combined_state[["scope", "entity_name", "customers", "avg_cltv", "churn_rate_pct", "revenue_at_risk", "ease_of_intervention_score", "priority_score", "priority_basis"]],
    combined_city[["scope", "entity_name", "customers", "avg_cltv", "churn_rate_pct", "revenue_at_risk", "ease_of_intervention_score", "priority_score", "priority_basis"]],
], ignore_index=True)

combined["priority_rank_within_scope"] = (
    combined.groupby("scope")["priority_score"].rank(method="first", ascending=False).astype(int)
)

print_result(
    f"Created a combined priority dataset containing {len(combined):,} entities across segments, states, and city markets."
)
display(combined.sort_values(["scope", "priority_score"], ascending=[True, False]).head(30))

Result: Created a combined priority dataset containing 1,110 entities across segments, states, and city markets.


,scope,entity_name,customers,avg_cltv,churn_rate_pct,revenue_at_risk,ease_of_intervention_score,priority_score,priority_basis,priority_rank_within_scope
4,city,California — San Diego,285,4345.09,64.91,385446.39,NaN,385446.39,Revenue at Risk only,1
5,city,California — Los Angeles,293,4369.85,26.62,147090.46,NaN,147090.46,Revenue at Risk only,2
6,city,California — Sacramento,108,4540.66,24.07,59449.20,NaN,59449.20,Revenue at Risk only,3
7,city,California — San Francisco,104,4335.46,29.81,56894.27,NaN,56894.27,Revenue at Risk only,4
8,city,California — Santa Barbara,28,4198.11,35.71,46033.71,NaN,46033.71,Revenue at Risk only,5
9,city,California — San Jose,112,4233.38,25.89,44393.49,NaN,44393.49,Revenue at Risk only,6
10,city,California — Temecula,38,4367.87,57.89,44102.41,NaN,44102.41,Revenue at Risk only,7
11,city,California — Escondido,51,4388.86,31.37,38869.89,NaN,38869.89,Revenue at Risk only,8
12,city,California — Fallbrook,43,4531.28,60.47,34691.78,NaN,34691.78,Revenue at Risk only,9
13,city,California — Torrance,25,4606.20,32.00,33534.79,NaN,33534.79,Revenue at Risk only,10


### Result & conclusion

I now have a common reporting structure for all business dimensions. I preserve the scope field so a segment priority cannot be mistaken for a geography priority.

In [11]:
segment_priority.to_csv(OUTPUT_DIR / "segment_priority.csv", index=False)
state_priority.to_csv(OUTPUT_DIR / "state_priority.csv", index=False)
city_priority.to_csv(OUTPUT_DIR / "city_priority.csv", index=False)
combined.to_csv(OUTPUT_DIR / "retention_strategy_priority.csv", index=False)

print_result("Saved segment, state, city, and combined prioritization outputs.")

Result: Saved segment, state, city, and combined prioritization outputs.


### Result & conclusion

The prioritization layer is now exported as reusable CSV artifacts. The combined file can be used directly by the next notebook and can also be loaded into Power BI or MySQL.